# VAE2 (Domain B) Training — Kaggle Notebook

Trains VAE2 on clean photos plus synthetically generated damage, independently of VAE1 -- these two can run in separate notebooks at the same time, since neither depends on the other's output. Only the translation network (stage 3) needs both finished checkpoints.

Includes validation support (`--val-clean-dir`/`--val-masks-dir`, `best.pt` saved whenever validation loss improves), four separate damage-type folders (scratches, smut, spots, dirt), and two numerical stability fixes added after an earlier run hit NaN loss: `logvar` is clamped in `models/vae.py` to prevent `exp(logvar)` overflow, and gradient clipping is applied in the training loop as a second layer of defense.

**Before running anything:**
1. Right sidebar: **Settings > Accelerator > GPU T4 x2**
2. Right sidebar: **Settings > Internet > On**


## 1. Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


CUDA available: True
GPU: Tesla T4


## 2. Install dependencies

In [2]:
!pip install -q pillow tqdm opencv-python-headless scikit-image scipy pandas
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


torch 2.10.0+cu128
torchvision 0.25.0+cu128


## 3. Recreate the project files

Includes the validation-enabled training script, the tested VAE2 architecture, and the NaN-prevention fixes (logvar clamping + gradient clipping).


In [3]:
import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)


In [4]:
%%writefile models/__init__.py



Writing models/__init__.py


In [5]:
%%writefile models/blocks.py
"""
Basic building blocks shared by the VAE encoder and decoder.

These are standard, well-known layer patterns (residual blocks, strided
conv downsampling, transposed-conv upsampling) -- not specific to any one
paper's architecture. The VAE class that assembles them into the actual
"Bringing Old Photos Back to Life"-style domain VAE is in vae.py, and that
assembly (encoder depth, bottleneck design, how mu/logvar are produced) is
the part you're implementing yourself from the paper's description.
"""

import torch
import torch.nn as nn


class ResidualBlock(nn.Module):
    """A standard two-conv residual block with instance normalization.
    Used inside the encoder/decoder to add capacity without changing
    spatial resolution."""

    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class DownsampleBlock(nn.Module):
    """Strided conv that halves spatial resolution and doubles channels
    (up to a cap), used to build the encoder's downsampling path."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    """Transposed conv that doubles spatial resolution, used to build the
    decoder's upsampling path (mirrors DownsampleBlock)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


Writing models/blocks.py


In [6]:
%%writefile models/vae.py
"""
A convolutional VAE with a *spatial* latent bottleneck (a small feature map,
not a single flattened vector) rather than the more familiar
flatten-to-a-vector VAE design.

Why spatial: "Bringing Old Photos Back to Life" needs the latent
representation to preserve rough spatial layout, so that later (in the
mapping network you'll build next) a damage mask can be used to tell the
model *where* in the latent space to focus repair. A flattened-vector
latent would throw that spatial correspondence away.

This same class is used for both VAE1 (domain A: real old photos) and VAE2
(domain B: clean photos) -- you'll instantiate two separate copies with
their own weights, one per domain, trained independently in
train_vae_domain_a.py / train_vae_domain_b.py.
"""

import torch
import torch.nn as nn

from models.blocks import ResidualBlock, DownsampleBlock, UpsampleBlock


class Encoder(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Initial conv, no downsampling yet
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, base_channels, kernel_size=7, padding=0),
            nn.InstanceNorm2d(base_channels, affine=True),
            nn.ReLU(inplace=True),
        ]

        # Downsampling path: halve spatial resolution each step, double
        # channels up to max_channels
        channels = base_channels
        for _ in range(n_downsample):
            next_channels = min(channels * 2, max_channels)
            layers.append(DownsampleBlock(channels, next_channels))
            channels = next_channels

        # Residual blocks at the bottleneck resolution, adding capacity
        # without further downsampling
        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        self.backbone = nn.Sequential(*layers)

        # Separate 1x1 convs producing the mean and log-variance maps of
        # the latent distribution -- same spatial size as the backbone
        # output, just a different channel count
        self.to_mu = nn.Conv2d(channels, latent_channels, kernel_size=1)
        self.to_logvar = nn.Conv2d(channels, latent_channels, kernel_size=1)

        self.bottleneck_channels = channels

    def forward(self, x: torch.Tensor):
        features = self.backbone(x)
        mu = self.to_mu(features)
        logvar = self.to_logvar(features)
        # Clamped here, at the source, so every downstream consumer
        # (reparameterize AND the KL loss) sees the same safe value --
        # without this, logvar can drift to a large value early in
        # training (before the encoder has learned sensible statistics),
        # and exp(logvar) genuinely overflows to inf, which corrupts
        # weights via the resulting huge gradient and collapses training
        # to NaN within the first few dozen batches. [-10, 10] is a
        # standard, well-established safe range: std = exp(0.5*logvar)
        # spans roughly 0.0067 to 148, wide enough to not meaningfully
        # constrain what the model can represent.
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, out_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Figure out the bottleneck channel count the same way the
        # encoder did, so the shapes line up
        channels = base_channels
        for _ in range(n_downsample):
            channels = min(channels * 2, max_channels)

        layers = [nn.Conv2d(latent_channels, channels, kernel_size=1)]

        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        # Upsampling path: mirror of the encoder's downsampling path
        channel_sequence = []
        c = base_channels
        for _ in range(n_downsample):
            channel_sequence.append(min(c * 2, max_channels))
            c = min(c * 2, max_channels)
        channel_sequence = [base_channels] + channel_sequence
        # channel_sequence e.g. [64, 128, 256, 512] for n_downsample=3;
        # we walk it backwards to go from bottleneck back to base_channels
        for i in range(n_downsample):
            in_ch = channel_sequence[n_downsample - i]
            out_ch = channel_sequence[n_downsample - i - 1]
            layers.append(UpsampleBlock(in_ch, out_ch))

        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(base_channels, out_channels, kernel_size=7, padding=0),
            nn.Tanh(),  # output in [-1, 1], matches how we'll normalize images
        ]

        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class DomainVAE(nn.Module):
    """
    Full VAE: encode -> reparameterize -> decode.

    Instantiate one of these per domain (real old photos / clean photos).
    The `Encoder`/`Decoder` above are shared *class* definitions but each
    DomainVAE instance gets its own independently-trained weights.
    """

    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)
        self.decoder = Decoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """The standard VAE reparameterization trick: sample z = mu + eps*std
        where eps ~ N(0, 1), so gradients can flow through the sampling step."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


def vae_loss(recon: torch.Tensor, target: torch.Tensor, mu: torch.Tensor,
             logvar: torch.Tensor, kl_weight: float = 1.0):
    """
    Standard VAE loss = reconstruction term + KL divergence term.

    Reconstruction uses L1 (tends to give sharper results than MSE for
    images -- this is a common choice in image-translation VAEs, not
    something unique to this paper).

    KL divergence pulls the latent distribution toward a standard normal,
    which is what makes the latent space smooth/well-structured enough for
    the mapping network to later translate between domains.
    """
    recon_loss = torch.nn.functional.l1_loss(recon, target)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + kl_weight * kl_loss
    return total_loss, recon_loss, kl_loss


def vae2_loss(recon_clean, recon_degraded, clean_target, mu_clean, logvar_clean,
              mu_degraded, logvar_degraded, kl_weight: float = 1.0, consistency_weight: float = 1.0):
    """
    VAE2's training objective is different from VAE1's plain reconstruction:
    it needs to learn a latent space where a clean photo AND a synthetically
    degraded version of it land close together, and BOTH decode back to the
    clean image. This is what lets the translation network later map a
    repaired latent through VAE2's decoder and get a clean-looking result.

    Four terms:
      1. Standard reconstruction of the clean branch (clean in -> clean out)
      2. Cross-reconstruction of the degraded branch (degraded in -> CLEAN
         out, not degraded out) -- this is the key difference from a plain
         autoencoder; it directly teaches "decode toward clean" regardless
         of which branch encoded the input
      3. Latent consistency: pulls the degraded branch's latent toward the
         clean branch's latent (clean side detached, so gradient flows
         into fixing the degraded encoder rather than both sides drifting
         together into a degenerate shortcut)
      4. KL divergence on both branches (standard VAE regularization)
    """
def vae2_loss(recon_clean, recon_degraded, clean_target, mu_clean, logvar_clean,
              mu_degraded, logvar_degraded, kl_weight: float = 1.0, consistency_weight: float = 1.0,
              mask=None, damage_weight: float = 5.0):
    """
    VAE2's training objective is different from VAE1's plain reconstruction:
    it needs to learn a latent space where a clean photo AND a synthetically
    degraded version of it land close together, and BOTH decode back to the
    clean image. This is what lets the translation network later map a
    repaired latent through VAE2's decoder and get a clean-looking result.

    Four terms:
      1. Standard reconstruction of the clean branch (clean in -> clean out)
      2. Cross-reconstruction of the degraded branch (degraded in -> CLEAN
         out, not degraded out) -- this is the key difference from a plain
         autoencoder; it directly teaches "decode toward clean" regardless
         of which branch encoded the input
      3. Latent consistency: pulls the degraded branch's latent toward the
         clean branch's latent (clean side detached, so gradient flows
         into fixing the degraded encoder rather than both sides drifting
         together into a degenerate shortcut)
      4. KL divergence on both branches (standard VAE regularization)

    If `mask` is provided (convention: 1.0 = clean, 0.0 = fully damaged),
    the degraded-branch reconstruction term (#2) is weighted to emphasize
    damaged pixels via `damage_weight` when computing `total_loss` (the
    value actually used for backward()) -- the same fix applied to Method
    3 and Method 2's Stage 1, and for the same reason: damaged pixels are
    typically a small fraction of the image, so this term can improve
    steadily just from the network getting better at reproducing the
    overall image, without specifically learning to repair damage. Only
    this term is weighted -- recon_loss_clean (#1) has no damage in its
    input/target pair at all, so there's nothing to weight there.

    The RETURNED recon_loss_degraded stays the plain, unweighted L1 value
    (comparable to pre-fix runs and to what evaluate.py measures) -- the
    weighted version is used only internally when building total_loss.
    """
    recon_loss_clean = torch.nn.functional.l1_loss(recon_clean, clean_target)
    recon_loss_degraded = torch.nn.functional.l1_loss(recon_degraded, clean_target)

    if mask is None:
        weighted_recon_degraded = recon_loss_degraded
    else:
        per_pixel_l1 = torch.abs(recon_degraded - clean_target)
        weight_map = 1.0 + damage_weight * (1.0 - mask)
        weighted_recon_degraded = (per_pixel_l1 * weight_map).mean()

    consistency_loss = torch.nn.functional.l1_loss(mu_degraded, mu_clean.detach())

    kl_clean = -0.5 * torch.mean(1 + logvar_clean - mu_clean.pow(2) - logvar_clean.exp())
    kl_degraded = -0.5 * torch.mean(1 + logvar_degraded - mu_degraded.pow(2) - logvar_degraded.exp())
    kl_loss = kl_clean + kl_degraded

    total_loss = (recon_loss_clean + weighted_recon_degraded
                  + consistency_weight * consistency_loss
                  + kl_weight * kl_loss)

    return total_loss, recon_loss_clean, recon_loss_degraded, consistency_loss, kl_loss


Writing models/vae.py


In [7]:
%%writefile data/__init__.py



Writing data/__init__.py


In [8]:
%%writefile data/vae2_pair_dataset.py
"""
Dataset for VAE2 (domain B) and the latent translation network: yields
(clean, degraded, mask) triples. Degraded images are composited on the fly
using your FilmDamageSimulator masks (screen blend by default -- see
composite_damage.py), same approach as the DiffBIR Stage 1 dataset.

Returning the raw mask (not just the composited degraded image) is what's
new here -- VAE2 training itself doesn't need it, but the translation
network trained on top of VAE2 does, since it uses the mask to know where
to apply heavier repair.
"""

import os
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF


class VAE2PairDataset(Dataset):
    def __init__(self, clean_dir: str, masks_dir, image_size: int = 256, augment: bool = True,
                 blend_mode: str = "screen", white_probability: float = 0.8):
        if blend_mode not in ("screen", "multiply", "mixed"):
            raise ValueError(f"blend_mode must be 'screen', 'multiply', or 'mixed', got '{blend_mode}'")
        if not (0.0 <= white_probability <= 1.0):
            raise ValueError(f"white_probability must be between 0 and 1, got {white_probability}")
        self.clean_dir = clean_dir
        self.masks_dirs = [masks_dir] if isinstance(masks_dir, str) else list(masks_dir)
        self.image_size = image_size
        self.augment = augment
        self.blend_mode = blend_mode
        self.white_probability = white_probability

        valid_ext = (".jpg", ".jpeg", ".png")
        self.clean_files = [f for f in os.listdir(clean_dir) if f.lower().endswith(valid_ext)]

        self.mask_files = []
        for d in self.masks_dirs:
            for f in os.listdir(d):
                if f.lower().endswith(".png") and not f.startswith("binarised_mask"):
                    self.mask_files.append(os.path.join(d, f))

        if len(self.clean_files) == 0:
            raise ValueError(f"No clean images found in {clean_dir}")
        if len(self.mask_files) == 0:
            raise ValueError(f"No usable masks found in {self.masks_dirs}")

        load_size = int(image_size * 1.12)
        self.clean_resize = T.Resize(load_size)
        self.image_size_final = image_size

    def __len__(self):
        return len(self.clean_files)

    def _load_clean(self, idx):
        path = os.path.join(self.clean_dir, self.clean_files[idx])
        img = Image.open(path).convert("RGB")
        return self.clean_resize(img)

    def _load_random_mask(self):
        path = random.choice(self.mask_files)
        return Image.open(path).convert("L")

    def _synchronized_crop_and_flip(self, clean_img, mask_img):
        mask_img = mask_img.resize(clean_img.size, Image.BILINEAR)

        if self.augment:
            i, j, h, w = T.RandomCrop.get_params(clean_img, output_size=(self.image_size_final, self.image_size_final))
            clean_img = TF.crop(clean_img, i, j, h, w)
            mask_img = TF.crop(mask_img, i, j, h, w)
            if random.random() < 0.5:
                clean_img = TF.hflip(clean_img)
                mask_img = TF.hflip(mask_img)
            if random.random() < 0.5:
                angle = random.choice([90, 180, 270])
                mask_img = TF.rotate(mask_img, angle)
        else:
            clean_img = TF.center_crop(clean_img, (self.image_size_final, self.image_size_final))
            mask_img = TF.center_crop(mask_img, (self.image_size_final, self.image_size_final))

        return clean_img, mask_img

    def __getitem__(self, idx):
        try:
            clean_img = self._load_clean(idx)
            mask_img = self._load_random_mask()
        except Exception:
            return self.__getitem__(random.randrange(len(self)))

        clean_img, mask_img = self._synchronized_crop_and_flip(clean_img, mask_img)

        clean_tensor = TF.to_tensor(clean_img)
        mask_tensor = TF.to_tensor(mask_img)

        # In "mixed" mode, each sample independently rolls screen vs.
        # multiply according to white_probability -- see the note in
        # common/degraded_pair_dataset.py for the full reasoning.
        if self.blend_mode == "mixed":
            sample_blend = "screen" if random.random() < self.white_probability else "multiply"
        else:
            sample_blend = self.blend_mode

        if sample_blend == "screen":
            degraded_tensor = 1.0 - (1.0 - clean_tensor) * mask_tensor
        else:
            degraded_tensor = clean_tensor * mask_tensor

        normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        clean_tensor = normalize(clean_tensor)
        degraded_tensor = normalize(degraded_tensor)
        # mask stays in [0, 1] -- it's used as a conditioning signal, not an
        # image to reconstruct, so it doesn't need the [-1, 1] normalization

        return {"clean": clean_tensor, "degraded": degraded_tensor, "mask": mask_tensor}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    return (tensor * 0.5 + 0.5).clamp(0, 1)


Writing data/vae2_pair_dataset.py


In [9]:
%%writefile train_vae_domain_b.py
"""
Train VAE2 (domain B) on clean photos + their synthetically damaged
versions. Unlike VAE1, this trains with a dual-branch objective: both the
clean image AND its degraded version get encoded, and BOTH must decode
back to the CLEAN target, with a consistency term pulling their latents
together. See models/vae.py:vae2_loss for the full reasoning.

Usage:
    python train_vae_domain_b.py --clean-dir ./voc_data --masks-dir ./generated_masks \
        --epochs 50 --batch-size 8 --image-size 256 --out-dir ./runs/vae_domain_b
"""

import argparse
import os
import time
from datetime import datetime

import torch
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.vae import DomainVAE, vae2_loss
from data.vae2_pair_dataset import VAE2PairDataset, denormalize


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    def __init__(self, log_path):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def save_comparison_grid(model, batch, out_path, device, max_images=6):
    """Saves [clean input | clean recon | degraded input | degraded->clean recon]
    so you can visually confirm both branches are converging toward the
    same clean output, which is the whole point of VAE2's training."""
    model.eval()
    with torch.no_grad():
        n = min(max_images, batch["clean"].shape[0])
        clean = batch["clean"][:n].to(device)
        degraded = batch["degraded"][:n].to(device)

        recon_clean, _, _ = model(clean)
        recon_degraded, _, _ = model(degraded)

        grid = torch.cat([
            denormalize(clean), denormalize(recon_clean),
            denormalize(degraded), denormalize(recon_degraded),
        ], dim=0)
        vutils.save_image(grid, out_path, nrow=n)
    model.train()


@torch.no_grad()
def run_validation(model, val_dataloader, device, kl_weight, consistency_weight, damage_weight=5.0):
    """Same vae2_loss objective as training, run with gradients disabled
    in eval mode. Returns the average of each loss component. Note: the
    returned/logged "loss" reflects the same mask-weighted total_loss used
    for training, so it's the right value for best.pt selection; "rd"
    (recon_degraded) stays the plain, comparable L1 value."""
    model.eval()
    totals = {"loss": 0.0, "rc": 0.0, "rd": 0.0, "cons": 0.0, "kl": 0.0}
    n_batches = 0
    for batch in val_dataloader:
        clean = batch["clean"].to(device, non_blocking=True)
        degraded = batch["degraded"].to(device, non_blocking=True)
        mask = batch["mask"].to(device, non_blocking=True)

        recon_clean, mu_clean, logvar_clean = model(clean)
        recon_degraded, mu_degraded, logvar_degraded = model(degraded)
        loss, rc, rd, cons, kl = vae2_loss(
            recon_clean, recon_degraded, clean, mu_clean, logvar_clean,
            mu_degraded, logvar_degraded, kl_weight=kl_weight,
            consistency_weight=consistency_weight, mask=mask, damage_weight=damage_weight,
        )
        totals["loss"] += loss.item(); totals["rc"] += rc.item()
        totals["rd"] += rd.item(); totals["cons"] += cons.item(); totals["kl"] += kl.item()
        n_batches += 1
    model.train()
    if n_batches == 0:
        return {k: None for k in totals}
    return {k: v / n_batches for k, v in totals.items()}


def main():
    parser = argparse.ArgumentParser(description="Train VAE2 on clean photos + synthetic degradation (domain B).")
    parser.add_argument("--clean-dir", type=str, required=True, help="folder of clean photos (e.g. VOC2012)")
    parser.add_argument("--val-clean-dir", type=str, default=None,
                         help="optional held-out validation clean-photo folder. If set, validation loss "
                              "is computed after every epoch and the best checkpoint is saved as 'best.pt'.")
    parser.add_argument("--val-masks-dir", type=str, default=None, nargs="+",
                         help="required alongside --val-clean-dir; same multi-folder format as --masks-dir")
    parser.add_argument("--masks-dir", type=str, required=True, nargs="+",
                         help="one or more mask folders; pass multiple to combine damage types kept "
                              "in separate folders, e.g. --masks-dir ./data/masks/scratches ./data/masks/smut")
    parser.add_argument("--blend-mode", type=str, choices=["screen", "multiply", "mixed"], default="screen")
    parser.add_argument("--white-probability", type=float, default=0.8,
                         help="only used with --blend-mode mixed; fraction of samples using screen (white) blend")
    parser.add_argument("--damage-weight", type=float, default=5.0,
                         help="extra loss weight applied to damaged pixels in the degraded-branch "
                              "reconstruction term -- without this, damaged pixels are typically a small "
                              "fraction of the image, so that term can improve steadily while barely "
                              "learning to repair damage. Set to 0 to disable and use plain unweighted L1 "
                              "(the old behavior). Only affects the degraded branch, not the clean branch.")
    parser.add_argument("--stop-after-epoch", type=int, default=1,
                         help="minimum epoch before early-stopping is even considered -- prevents "
                              "stopping on a lucky early epoch before training has genuinely converged. "
                              "Only used together with --stop-loss-below.")
    parser.add_argument("--stop-loss-below", type=float, default=None,
                         help="if set, training stops early once validation loss (the same 'loss' value "
                              "used for best.pt selection) drops below this threshold, checked at epoch "
                              ">= --stop-after-epoch. Requires --val-clean-dir to be set (validation must "
                              "be running for this to have anything to check). Leave unset to disable.")
    parser.add_argument("--patience", type=int, default=None,
                         help="if set, training stops early if validation loss hasn't improved on the "
                              "best-so-far value for this many CONSECUTIVE validation checks -- catches "
                              "divergence/overfitting of any kind (e.g. KL or consistency terms blowing "
                              "up even while reconstruction quality holds steady), not just a fixed "
                              "absolute threshold like --stop-loss-below. Also respects "
                              "--stop-after-epoch. Leave unset to disable.")
    parser.add_argument("--out-dir", type=str, default="./runs/vae_domain_b")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--lr-schedule", type=str, choices=["none", "cosine"], default="none",
                         help="'cosine' decays the learning rate smoothly from --lr down to near-zero "
                              "over the full --epochs range. Addresses a different failure mode than "
                              "--kl-weight: a fixed learning rate held constant across many epochs can "
                              "gradually destabilize the KL term late in training, even when reconstruction "
                              "quality is still improving. 'none' (default) keeps the old fixed-LR behavior.")
    parser.add_argument("--kl-weight", type=float, default=0.01)
    parser.add_argument("--kl-anneal-epochs", type=int, default=0,
                         help="ramp kl_weight linearly from 0 to --kl-weight over this many "
                              "epochs, instead of applying the full weight from epoch 1. "
                              "Lets reconstruction stabilize before KL pressure kicks in -- "
                              "targets the KL-divergence-wall failure mode directly, rather "
                              "than just retuning a fixed kl_weight value. 0 disables annealing "
                              "(same behavior as before).")
    parser.add_argument("--consistency-weight", type=float, default=1.0,
                         help="weight pulling the degraded branch's latent toward the clean branch's latent")
    parser.add_argument("--latent-channels", type=int, default=64)
    parser.add_argument("--n-downsample", type=int, default=3)
    parser.add_argument("--n-residual-blocks", type=int, default=4)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--sample-every", type=int, default=200)
    parser.add_argument("--steps-per-epoch", type=int, default=None)
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--val-every", type=int, default=1,
                         help="run validation every N epochs (only used if --val-clean-dir is set)")
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--amp", action="store_true")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")
    cpu_count = os.cpu_count()
    log(f"  CPUs available: {cpu_count}, --num-workers set to {args.num_workers}")

    log("Building dataset index...")
    dataset = VAE2PairDataset(args.clean_dir, args.masks_dir, image_size=args.image_size,
                               augment=True, blend_mode=args.blend_mode, white_probability=args.white_probability)
    log(f"Loaded {len(dataset)} clean images, {len(dataset.mask_files)} masks "
        f"(blend_mode={args.blend_mode})")

    if args.steps_per_epoch:
        num_samples = args.steps_per_epoch * args.batch_size
        sampler = RandomSampler(dataset, replacement=True, num_samples=num_samples)
        dataloader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))
        log(f"Using --steps-per-epoch {args.steps_per_epoch}: {num_samples} images/epoch")
    else:
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))

    log("Fetching a fixed sample batch for visualization...")
    fixed_batch = next(iter(dataloader))
    log("Dataset ready.")

    val_dataloader = None
    if args.val_clean_dir:
        if not args.val_masks_dir:
            raise ValueError("--val-clean-dir requires --val-masks-dir")
        log(f"Building validation dataset from {args.val_clean_dir} / {args.val_masks_dir}...")
        val_dataset = VAE2PairDataset(args.val_clean_dir, args.val_masks_dir, image_size=args.image_size,
                                       augment=False, blend_mode=args.blend_mode, white_probability=args.white_probability)
        val_dataloader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False,
                                     num_workers=args.num_workers, drop_last=False,
                                     pin_memory=(device.type == "cuda"))
        log(f"Loaded {len(val_dataset)} validation clean images")

    best_val_loss = float("inf")
    epochs_since_improvement = 0

    model = DomainVAE(
        in_channels=3, n_downsample=args.n_downsample,
        n_residual_blocks=args.n_residual_blocks, latent_channels=args.latent_channels,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.5, 0.999))

    if args.lr_schedule == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    else:
        scheduler = None

    use_amp = args.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)
        if scheduler is not None:
            # Deliberately NOT loading a saved scheduler_state_dict here.
            # CosineAnnealingLR's state includes T_max-dependent values --
            # if this run's --epochs differs from the original run's (e.g.
            # extending training from 50 to 80 epochs, as is common when
            # resuming), loading a state_dict built for the OLD T_max into
            # a scheduler built for the NEW T_max produces incorrect,
            # non-monotonic LR values. Manually fast-forwarding against
            # the CURRENT T_max is correct regardless of whether --epochs
            # changed between runs.
            # CosineAnnealingLR computes each step RECURSIVELY from the
            # CURRENT lr in optimizer.param_groups, not purely from
            # last_epoch -- so optimizer.load_state_dict() just above,
            # which restores the checkpoint's already-decayed lr, would
            # corrupt the fast-forward if left in place. Reset to the base
            # lr first so the recursive stepping starts from the correct
            # baseline, matching what a genuinely fresh scheduler would do.
            for group in optimizer.param_groups:
                group["lr"] = args.lr
            log(f"  Fast-forwarding cosine schedule (T_max={args.epochs}) to epoch {start_epoch - 1}")
            for _ in range(start_epoch - 1):
                scheduler.step()

    num_params = sum(p.numel() for p in model.parameters())
    log(f"Model has {num_params:,} parameters")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running_loss = 0.0
        running_rc, running_rd, running_cons, running_kl = 0.0, 0.0, 0.0, 0.0

        if args.kl_anneal_epochs > 0:
            effective_kl_weight = args.kl_weight * min(1.0, epoch / args.kl_anneal_epochs)
        else:
            effective_kl_weight = args.kl_weight

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False)
        batch_end_time = time.time()

        for batch in progress_bar:
            data_time = time.time() - batch_end_time
            compute_start = time.time()

            clean = batch["clean"].to(device, non_blocking=True)
            degraded = batch["degraded"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.amp.autocast(device.type, enabled=use_amp):
                recon_clean, mu_clean, logvar_clean = model(clean)
                recon_degraded, mu_degraded, logvar_degraded = model(degraded)
                loss, rc, rd, cons, kl = vae2_loss(
                    recon_clean, recon_degraded, clean, mu_clean, logvar_clean,
                    mu_degraded, logvar_degraded, kl_weight=effective_kl_weight,
                    consistency_weight=args.consistency_weight,
                    mask=mask, damage_weight=args.damage_weight,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            if device.type == "cuda":
                torch.cuda.synchronize()
            compute_time = time.time() - compute_start

            running_loss += loss.item()
            running_rc += rc.item()
            running_rd += rd.item()
            running_cons += cons.item()
            running_kl += kl.item()
            global_step += 1

            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

            if args.log_every and global_step % args.log_every == 0:
                log(f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s")

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_comparison_grid(model, fixed_batch, sample_path, device)
                log(f"  Saved sample grid: {sample_path}")

            batch_end_time = time.time()

        n_batches = len(dataloader)
        elapsed = time.time() - start_time
        current_lr = optimizer.param_groups[0]["lr"]
        log(f"[Epoch {epoch}/{args.epochs}] loss={running_loss / n_batches:.4f} "
            f"recon_clean={running_rc / n_batches:.4f} recon_degraded={running_rd / n_batches:.4f} "
            f"consistency={running_cons / n_batches:.4f} kl={running_kl / n_batches:.4f} "
            f"kl_weight={effective_kl_weight:.5f} "
            f"lr={current_lr:.6f} "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if scheduler is not None:
            scheduler.step()

        if val_dataloader is not None and epoch % args.val_every == 0:
            val = run_validation(model, val_dataloader, device, effective_kl_weight, args.consistency_weight,
                                  damage_weight=args.damage_weight)
            log(f"  [Validation] epoch {epoch}: loss={val['loss']:.4f} recon_clean={val['rc']:.4f} "
                f"recon_degraded={val['rd']:.4f} consistency={val['cons']:.4f} kl={val['kl']:.4f}")
            if val["loss"] < best_val_loss:
                best_val_loss = val["loss"]
                epochs_since_improvement = 0
                best_path = os.path.join(checkpoints_dir, "best.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "val_loss": val["loss"],
                    "args": vars(args),
                }, best_path)
                log(f"  New best validation loss ({val['loss']:.4f}) -- saved {best_path}")
            else:
                epochs_since_improvement += 1

            if (args.stop_loss_below is not None and epoch >= args.stop_after_epoch
                    and val["loss"] < args.stop_loss_below):
                log(f"  Validation loss ({val['loss']:.4f}) dropped below --stop-loss-below "
                    f"({args.stop_loss_below}) at epoch {epoch} (>= --stop-after-epoch "
                    f"{args.stop_after_epoch}) -- stopping early.")
                ckpt_path = os.path.join(checkpoints_dir, f"vae_domain_b_epoch{epoch:04d}.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "args": vars(args),
                }, ckpt_path)
                log(f"  Saved final checkpoint before stopping: {ckpt_path}")
                break

            if (args.patience is not None and epoch >= args.stop_after_epoch
                    and epochs_since_improvement >= args.patience):
                log(f"  Validation loss hasn't improved on the best value ({best_val_loss:.4f}) for "
                    f"{epochs_since_improvement} consecutive checks (>= --patience {args.patience}) "
                    f"at epoch {epoch} -- stopping early. This catches divergence even when "
                    f"reconstruction quality (recon_clean/recon_degraded) looks fine, since it's "
                    f"driven by the combined validation loss, not just one component.")
                ckpt_path = os.path.join(checkpoints_dir, f"vae_domain_b_epoch{epoch:04d}.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "args": vars(args),
                }, ckpt_path)
                log(f"  Saved final checkpoint before stopping: {ckpt_path}")
                log(f"  Note: best.pt (epoch with val_loss={best_val_loss:.4f}) is likely more useful "
                    f"than this final checkpoint for downstream use, given training had stopped "
                    f"improving.")
                break

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"vae_domain_b_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch, "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    log(f"Training complete. Total time: {format_duration(time.time() - start_time)}")
    logger.close()


if __name__ == "__main__":
    main()

Writing train_vae_domain_b.py


In [10]:
%%writefile composite_damage.py
"""
Composite a generated damage mask (from generate_synthetic_only.py or
damage_generator.py) onto a clean target image, producing a damaged/clean
training pair for restoration model training.

The mask convention from this codebase: 255 = clean/undamaged, values toward
0 = damaged (dust, dirt, scratches etc).

Three blend modes are supported:
  - "screen": LIGHTENS toward white at damaged pixels. This is the
    physically realistic choice for most scratch/abrasion damage, where the
    print's emulsion is scraped away and the lighter paper base shows
    through -- old photo scratches are usually bright/white marks, not dark
    ones.
  - "multiply": DARKENS toward black at damaged pixels. More appropriate for
    damage types that genuinely deposit dark material (soot/smut, heavy
    dirt, mold staining) rather than abrading the surface.
  - "mixed" (default): randomly picks screen or multiply for THIS composite,
    weighted by --white-probability (default 0.8 = 80% chance of white
    scratches-style damage, 20% chance of black smut-style damage). Since
    this tool composites one image at a time, running it repeatedly (e.g.
    in a loop over many images) with "mixed" will produce that ratio across
    the batch, rather than a single fixed appearance for every image.

Since a single generated mask can currently mix multiple damage types
(e.g. scratches + smut) without tracking which pixel came from which type,
blend selection here is a per-composite choice rather than automatic
per-pixel selection.

Usage:
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend multiply
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend mixed --white-probability 0.8
"""

import argparse
import random

import cv2 as cv
import numpy as np


def composite(clean_img, mask_img, blend="screen", white_probability=0.8):
    if clean_img.shape[:2] != mask_img.shape[:2]:
        mask_img = cv.resize(mask_img, (clean_img.shape[1], clean_img.shape[0]), interpolation=cv.INTER_LINEAR)

    mask_norm = mask_img.astype(np.float32) / 255.0
    if clean_img.ndim == 3 and mask_norm.ndim == 2:
        mask_norm = mask_norm[:, :, None]

    clean_f = clean_img.astype(np.float32)

    resolved_blend = blend
    if blend == "mixed":
        resolved_blend = "screen" if random.random() < white_probability else "multiply"

    if resolved_blend == "screen":
        # Lightens toward white at damaged (low-mask) pixels.
        damaged = 255.0 - (255.0 - clean_f) * mask_norm
    elif resolved_blend == "multiply":
        # Darkens toward black at damaged (low-mask) pixels.
        damaged = clean_f * mask_norm
    else:
        raise ValueError(f"Unknown blend mode '{blend}', expected 'screen', 'multiply', or 'mixed'")

    # Kept as a single return value (not a tuple) so existing callers that
    # do `damaged_img = composite(clean, mask)` keep working unchanged --
    # the resolved mode is attached as an attribute instead, for callers
    # that want to know which mode was actually picked in "mixed" mode.
    result = np.clip(damaged, 0, 255).astype(np.uint8)
    composite.last_resolved_blend = resolved_blend
    return result


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Composite a damage mask onto a clean image.')
    parser.add_argument('--clean', required=True, help='path to the clean input image')
    parser.add_argument('--mask', required=True, help='path to the generated grayscale damage mask')
    parser.add_argument('--out', required=True, help='path to write the damaged output image')
    parser.add_argument('--blend', choices=['screen', 'multiply', 'mixed'], default='screen',
                         help="'screen' (default) always produces light/white damage marks; "
                              "'multiply' always produces dark damage marks; "
                              "'mixed' randomly picks screen/multiply per --white-probability")
    parser.add_argument('--white-probability', type=float, default=0.8,
                         help="only used with --blend mixed; probability of picking screen (white) "
                              "over multiply (black) for this composite (default 0.8)")
    args = parser.parse_args()

    clean_img = cv.imread(args.clean, cv.IMREAD_UNCHANGED)
    mask_img = cv.imread(args.mask, cv.IMREAD_GRAYSCALE)

    damaged = composite(clean_img, mask_img, blend=args.blend, white_probability=args.white_probability)
    cv.imwrite(args.out, damaged)
    print(f"Wrote damaged image to {args.out} (blend={args.blend}, resolved to '{composite.last_resolved_blend}')")


Writing composite_damage.py


## 4. Get VOC2012 clean images — automatic, no manual download

In [11]:
import torchvision.datasets as tvds

VOC_ROOT = '/kaggle/working/voc_data'
VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')

if os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0:
    print(f'VOC2012 already present at {VOC_JPEG_DIR}, skipping download.')
else:
    os.makedirs(VOC_ROOT, exist_ok=True)
    _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)

num_images = len([f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')])
print(f'VOC2012 ready: {num_images} images at {VOC_JPEG_DIR}')


100%|██████████| 2.00G/2.00G [01:12<00:00, 27.6MB/s]


VOC2012 ready: 17125 images at /kaggle/working/voc_data/VOCdevkit/VOC2012/JPEGImages


## 4.5. Build a small validation subset

A separate slice of VOC2012, disjoint from what training uses, for `--val-clean-dir` below.


In [12]:
import random, shutil

VAL_SUBSET_DIR = '/kaggle/working/voc_val_subset/images'
os.makedirs(VAL_SUBSET_DIR, exist_ok=True)

existing_val = [f for f in os.listdir(VAL_SUBSET_DIR) if f.lower().endswith('.jpg')]
TARGET_N_VAL = 150

if len(existing_val) >= TARGET_N_VAL:
    print(f'{len(existing_val)} validation images already present, skipping.')
else:
    all_voc_images = [f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')]
    random.seed(42)
    val_subset = random.sample(all_voc_images, TARGET_N_VAL)
    for fname in val_subset:
        shutil.copy(os.path.join(VOC_JPEG_DIR, fname), os.path.join(VAL_SUBSET_DIR, fname))
    print(f'Copied {len(val_subset)} validation images to {VAL_SUBSET_DIR}')


Copied 150 validation images to /kaggle/working/voc_val_subset/images


## 5. Generate damage masks

Four damage types, each into its OWN folder -- combine whichever subset you want per run via `--masks-dir`. Adjust `MIN_COUNT`/`MAX_COUNT`/`SIZE_VARIETY_MIN`/`SIZE_VARIETY_MAX` and re-run if a type looks too faint or too heavy once you check the composited samples in Section 6.


In [13]:
# if os.path.isdir('FilmDamageSimulator'):
#     print('FilmDamageSimulator already cloned, skipping.')
# else:
#     !git clone --depth 1 https://github.com/daniela997/FilmDamageSimulator.git


In [14]:
# %%writefile FilmDamageSimulator/damage_generator/generate_synthetic_only.py
# """
# Generate damage overlay masks using ONLY the pre-classified synthetic damage
# patches in /synthetic/<type>/ (e.g. scratches, smut), without ever touching
# the real scanned film frames in /scans/.

# This bypasses damage_generator.py's default behaviour, which always loads
# /scans/ and mixes real scanned artifact crops into the sampling pool even
# when --synthetic is passed. Here, only the folder(s) you name are loaded,
# and artifact count/size statistics are fit on those patches' own area
# distribution instead of the real-scan-derived Gamma distributions.

# Usage:
#     python generate_synthetic_only.py --types scratches,smut --height 1024 --width 1024
#     python generate_synthetic_only.py --types scratches --procedural-scratches
#     python generate_synthetic_only.py --types dirt --out-dir ./masks/dirt --size-variety-min 0.5 --size-variety-max 1.8
# """

# import os
# import argparse
# import uuid
# import random
# import numpy as np
# import pandas as pd
# import cv2 as cv
# import scipy.stats as stats
# import skimage.transform as skimage_tf

# from scans import load_images
# from generate_masks import generate_perlin_noise_2d, increase_contrast, random_perlin_with_numpy, line_scratch


# def sample_size_from_own_distribution(df, num_artifact):
#     """Fit a Gamma distribution to this dataframe's OWN artifact areas
#     (instead of a real-scan-derived one) and sample target sizes from it."""
#     areas = df['Contour Area']
#     gamma_param = stats.gamma.fit(areas, floc=0)
#     shape, _, scale = gamma_param
#     return np.random.gamma(shape, scale, num_artifact)


# def sample_closest_in_area(df, target_areas):
#     df = df.sample(frac=1).reset_index(drop=True)
#     areas = df['Contour Area']
#     indexes = []
#     for target in target_areas:
#         candidates = df.iloc[(areas - target).abs().argsort()[:15]].index.tolist()
#         index = random.choice(candidates)
#         indexes.append(index)
#         areas = areas.drop(areas.index[[index]])
#     picked = df.iloc[indexes].copy()
#     picked['Target size'] = target_areas
#     return picked


# def build_mask(target_size, per_type_dfs, per_type_counts, rescale=True, verbose=False,
#                 size_variety=(0.5, 1.8)):
#     rescale_factor = (target_size[0] / 2560 if target_size[0] <= target_size[1]
#                        else target_size[1] / 2560) if rescale else 1.

#     selected_frames = []
#     for artifact_type, df in per_type_dfs.items():
#         lo, hi = per_type_counts[artifact_type]
#         num = int(np.random.randint(lo, hi + 1))
#         if num == 0 or len(df) == 0:
#             continue
#         target_areas = sample_size_from_own_distribution(df, num)
#         picked = sample_closest_in_area(df, target_areas)
#         selected_frames.append(picked)
#         if verbose:
#             print(f"Selected {num} '{artifact_type}' artifacts")

#     if not selected_frames:
#         raise ValueError("No artifacts selected - check your --types and --min-count/--max-count")

#     selected_artifacts_df = pd.concat(selected_frames, ignore_index=True)
#     artifacts_num = len(selected_artifacts_df)

#     mask_final = np.zeros(target_size).astype(np.uint8)
#     perlin_noise = generate_perlin_noise_2d(target_size, (2, 2))
#     normalised_noise = (perlin_noise - np.min(perlin_noise)) / np.ptp(perlin_noise)
#     xs, ys = random_perlin_with_numpy(artifacts_num, normalised_noise)
#     random_angles = np.random.randint(0, 360, size=artifacts_num)

#     i = 0
#     for _, artifact_row in selected_artifacts_df.iterrows():
#         try:
#             artifact = artifact_row['Artifact'].astype(np.uint8)
#             random_scale = artifact_row['Target size'] / artifact_row['Contour Area']
#             random_angle = random_angles[i]
#             # size_variety adds an independent random multiplier on top of the
#             # gamma-fit-based scale above -- without this, size variety is
#             # entirely bounded by whatever area distribution the raw artifact
#             # patches happen to have, which can look narrower than intended if
#             # the patch library itself is fairly uniform in size.
#             variety_factor = np.random.uniform(size_variety[0], size_variety[1])
#             new_rescale_factor = rescale_factor * np.sqrt(random_scale) * variety_factor
#             artifact = skimage_tf.rescale(artifact, round(new_rescale_factor, 2), anti_aliasing=True, preserve_range=True)
#             artifact = skimage_tf.rotate(artifact, angle=random_angle, resize=True, preserve_range=True)
#             artifact_w, artifact_h = artifact.shape[:2]

#             x1 = xs[i] - artifact_w // 2
#             x2 = x1 + artifact_w
#             if x1 < 0:
#                 artifact = artifact[-x1:, :]; x1 = 0
#             if x2 > target_size[0]:
#                 artifact = artifact[:-(x2 - target_size[0]), :]; x2 = target_size[0]

#             y1 = ys[i] - artifact_h // 2
#             y2 = y1 + artifact_h
#             if y1 < 0:
#                 artifact = artifact[:, -y1:]; y1 = 0
#             if y2 > target_size[1]:
#                 artifact = artifact[:, :-(y2 - target_size[1])]; y2 = target_size[1]

#             mask_final[x1:x2, y1:y2] = np.where(
#                 artifact > mask_final[x1:x2, y1:y2], artifact, mask_final[x1:x2, y1:y2]
#             )
#             i += 1
#         except Exception:
#             i += 1
#             continue

#     mask_final = np.invert(mask_final.astype(np.uint8))
#     binarised = ((mask_final > 240) * 255).astype(np.uint8)
#     return mask_final.astype(np.uint8), binarised


# def add_procedural_scratches(mask, height, width, verbose=False):
#     """Blend in fully procedural (Perlin-noise-based) scratch lines.
#     These require NO source images at all -- real or synthetic -- so they
#     are always 'safe' to include without pulling in any scan data."""
#     num_extra_scratch = int(np.random.gamma(6, 2, 1)[0])
#     for _ in range(num_extra_scratch):
#         length = np.random.randint(10, high=max(height, width), dtype=int)
#         try:
#             scratch = line_scratch(np.array(length))
#             sw, sh = scratch.shape[:2]
#             if sw >= width or sh >= height:
#                 continue
#             x1 = np.random.randint(0, width - sw)
#             y1 = np.random.randint(0, height - sh)
#             region = mask[x1:x1 + sw, y1:y1 + sh]
#             mask[x1:x1 + sw, y1:y1 + sh] = np.minimum(region, np.invert(scratch.astype(np.uint8)))
#         except Exception:
#             continue
#     if verbose:
#         print(f"Added {num_extra_scratch} procedural scratch lines")
#     return mask


# if __name__ == '__main__':
#     parser = argparse.ArgumentParser(
#         description='Generate damage masks from ONLY classified synthetic patches (no scanned frames).'
#     )
#     parser.add_argument('--types', type=str, default='scratches,smut',
#                          help='comma-separated subfolder names under /synthetic/, '
#                               'e.g. scratches,smut,dirt,dots,hair,hair-short,lint,sprinkles,spots,stain')
#     parser.add_argument('--height', type=int, default=1024)
#     parser.add_argument('--width', type=int, default=1024)
#     parser.add_argument('--min-count', type=int, default=3, help='min number of artifacts per type')
#     parser.add_argument('--max-count', type=int, default=15, help='max number of artifacts per type')
#     parser.add_argument('--size-variety-min', type=float, default=0.5,
#                          help="minimum extra random size multiplier applied per artifact, independent of "
#                               "the patch library's own area distribution -- lower values allow smaller "
#                               "damage instances")
#     parser.add_argument('--size-variety-max', type=float, default=1.8,
#                          help="maximum extra random size multiplier applied per artifact -- higher values "
#                               "allow larger damage instances")
#     parser.add_argument('--procedural-scratches', action='store_true',
#                          help='also blend in fully procedural line scratches (no source image needed)')
#     parser.add_argument('--n', type=int, default=1, help='how many masks to generate')
#     parser.add_argument('--out-dir', type=str, default=None,
#                          help='where to write masks. Defaults to <repo_root>/generated/. Set this explicitly '
#                               'to generate straight into a per-type folder, e.g. --out-dir ../../data/masks/scratches '
#                               'when running with --types scratches only, so different damage types land in '
#                               'physically separate folders instead of one mixed pool.')
#     parser.add_argument('--verbose', action='store_true')
#     args = parser.parse_args()

#     abs_path = os.path.abspath(os.path.dirname(__file__))
#     synthetic_path = os.path.dirname(os.path.normpath(abs_path)) + '/synthetic/'
#     out_dir = args.out_dir if args.out_dir else os.path.dirname(os.path.normpath(abs_path)) + '/generated/'
#     if not out_dir.endswith('/'):
#         out_dir += '/'
#     os.makedirs(out_dir, exist_ok=True)

#     types = [t.strip() for t in args.types.split(',') if t.strip()]

#     per_type_dfs = {}
#     for t in types:
#         df = load_images(synthetic_path, t, verbose=args.verbose)
#         df['Contour Area'] = df['Non-zero pixel area']
#         per_type_dfs[t] = df
#         print(f"Loaded {len(df)} '{t}' artifact patches from /synthetic/{t}/")

#     per_type_counts = {t: (args.min_count, args.max_count) for t in types}

#     for n in range(args.n):
#         mask, binary_mask = build_mask(
#             (args.height, args.width), per_type_dfs, per_type_counts, verbose=args.verbose,
#             size_variety=(args.size_variety_min, args.size_variety_max)
#         )

#         if args.procedural_scratches:
#             mask = add_procedural_scratches(mask, args.height, args.width, verbose=args.verbose)
#             binary_mask = ((mask > 240) * 255).astype(np.uint8)

#         uid = str(uuid.uuid4())[:8]
#         tag = "_".join(types)
#         cv.imwrite(out_dir + f'mask_{tag}_{uid}.png', mask)
#         cv.imwrite(out_dir + f'binarised_mask_{tag}_{uid}.png', binary_mask)
#         print(f"[{n+1}/{args.n}] Saved mask_{tag}_{uid}.png")

#     print(f"Done. Masks written to {out_dir}")


In [15]:
# DAMAGE_TYPES = ['scratches', 'smut', 'spots', 'dirt']
# TARGET_N_MASKS = 3000  # per type
# MIN_COUNT = 3
# MAX_COUNT = 15
# SIZE_VARIETY_MIN = 0.5
# SIZE_VARIETY_MAX = 1.8

# os.chdir('/kaggle/working/FilmDamageSimulator/damage_generator')

# MASKS_DIRS = []
# for damage_type in DAMAGE_TYPES:
#     type_dir = f'/kaggle/working/generated_masks/{damage_type}'
#     os.makedirs(type_dir, exist_ok=True)
#     existing = [f for f in os.listdir(type_dir) if f.startswith('mask_')]

#     if len(existing) >= TARGET_N_MASKS:
#         print(f'[{damage_type}] {len(existing)} masks already present, skipping.')
#     else:
#         needed = TARGET_N_MASKS - len(existing)
#         print(f'[{damage_type}] generating {needed} masks...')
#         abs_type_dir = os.path.abspath(type_dir)
#         !python generate_synthetic_only.py --types {damage_type} \
#             --height 128 --width 128 --min-count {MIN_COUNT} --max-count {MAX_COUNT} \
#             --size-variety-min {SIZE_VARIETY_MIN} --size-variety-max {SIZE_VARIETY_MAX} \
#             --n {needed} --out-dir {abs_type_dir} --verbose

#     MASKS_DIRS.append(type_dir)

# os.chdir('/kaggle/working')

# for d in MASKS_DIRS:
#     n = len([f for f in os.listdir(d) if f.startswith('mask_')]) if os.path.isdir(d) else 0
#     print(f'{d}: {n} masks')


In [16]:
import os

MASKS_DIR = '/kaggle/input/datasets/dorast/generated-masks'

DAMAGE_TYPES = ['dirt', 'scratches', 'smut', 'spots']  # adjust if the ls output above shows different names

MASKS_DIRS = [os.path.join(MASKS_DIR, t) for t in DAMAGE_TYPES]

for d in MASKS_DIRS:
    n = len([f for f in os.listdir(d) if f.startswith('mask_')]) if os.path.isdir(d) else 0
    print(f'{d}: {n} masks')

/kaggle/input/datasets/dorast/generated-masks/dirt: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/scratches: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/smut: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/spots: 3000 masks


## 6. Preview damage across all four types

One row per type, a few samples each, composited onto a real clean photo -- so you can judge intensity directly before committing to a full training run.


In [17]:
#  import matplotlib.pyplot as plt
#  import cv2 as cv
# import random
#  from composite_damage import composite

#  N_PREVIEW_SAMPLES = 3
#  preview_clean_path = os.path.join(VOC_JPEG_DIR, os.listdir(VOC_JPEG_DIR)[0])
#  clean_img = cv.imread(preview_clean_path)
#  clean_img = cv.resize(clean_img, (128, 128))

#  fig, axes = plt.subplots(len(DAMAGE_TYPES), N_PREVIEW_SAMPLES + 1,
#                            figsize=(3 * (N_PREVIEW_SAMPLES + 1), 3 * len(DAMAGE_TYPES)))

# for row, (damage_type, mask_dir) in enumerate(zip(DAMAGE_TYPES, MASKS_DIRS)):
#      mask_files = [f for f in os.listdir(mask_dir) if f.startswith('mask_')]
#      sample_masks = random.sample(mask_files, min(N_PREVIEW_SAMPLES, len(mask_files)))

#      axes[row, 0].imshow(cv.cvtColor(clean_img, cv.COLOR_BGR2RGB))
#      axes[row, 0].set_ylabel(damage_type, fontsize=13)
#      axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
#      if row == 0:
#         axes[row, 0].set_title('Clean')

#      for col, mask_fname in enumerate(sample_masks, start=1):
#          mask_img = cv.imread(os.path.join(mask_dir, mask_fname), cv.IMREAD_GRAYSCALE)
#         damaged_img = composite(clean_img, mask_img)
#         axes[row, col].imshow(cv.cvtColor(damaged_img, cv.COLOR_BGR2RGB))
#         axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
#         if row == 0:
#             axes[row, col].set_title(f'Sample {col}')

#  plt.tight_layout()
#  plt.show()


## 7. Quick smoke test (small config, low resolution, WITH validation)

In [18]:
# !python train_vae_domain_b.py \
#     --clean-dir "$VOC_JPEG_DIR" --val-clean-dir "$VAL_SUBSET_DIR" \
#     --masks-dir $MASKS_DIRS --val-masks-dir $MASKS_DIRS \
#     --epochs 3 --batch-size 8 --image-size 64 --num-workers 2 \
#     --n-downsample 3 --latent-channels 32 \
#     --steps-per-epoch 10 --log-every 5 --sample-every 10 --save-every 1 --val-every 1 \
#     --amp --out-dir ./runs/vae_domain_b_smoke_test --device cuda


## 8. View a reconstruction sample

Rows: clean input | clean reconstruction | degraded input | degraded→clean reconstruction.


In [19]:
# import glob
# from PIL import Image

# sample_files = sorted(glob.glob('runs/vae_domain_b_smoke_test/samples/*.png'))
# if sample_files:
#     img = Image.open(sample_files[-1])
#     plt.figure(figsize=(14, 10))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.title(f'Latest sample: {sample_files[-1]}')
#     plt.show()
# else:
#     print('No samples found yet — check the training cell above ran successfully.')


## 9. Baseline sanity check: short run WITH validation

Real architecture size, `--image-size 128`. Watch for the `[Validation]` log lines and confirm loss decreases smoothly across epochs without jumping to `nan` -- the logvar clamp and gradient clipping fixes should prevent the NaN failure mode seen in earlier VAE1 runs, but this is still worth watching on a fresh run.


In [20]:
# import subprocess

# result = subprocess.run([
#     'python', 'train_vae_domain_b.py',
#     '--clean-dir', VOC_JPEG_DIR, '--val-clean-dir', VAL_SUBSET_DIR,
#     '--masks-dir', *MASKS_DIRS, '--val-masks-dir', *MASKS_DIRS,
#     '--epochs', '2', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
#     '--n-downsample', '3', '--latent-channels', '64',
#     '--val-every', '1', '--log-every', '20', '--sample-every', '50', '--save-every', '4',
#     '--amp', '--out-dir', './runs/baseline_check', '--device', 'cuda',
# ])
# result.check_returncode()


In [21]:
# import subprocess

# result = subprocess.run([
#     'python', 'train_vae_domain_b.py',
#     '--clean-dir', VOC_JPEG_DIR, '--val-clean-dir', VAL_SUBSET_DIR,
#     '--masks-dir', *MASKS_DIRS, '--val-masks-dir', *MASKS_DIRS,
#     '--epochs', '30', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
#     '--n-downsample', '3', '--latent-channels', '64',
#     '--stop-loss-below', '0.05', '--patience', '3', '--stop-after-epoch', '3',
#     '--val-every', '1', '--log-every', '20', '--sample-every', '50', '--save-every', '4',
#     '--amp', '--out-dir', './runs/baseline_check', '--device', 'cuda',
#     '--resume', '/kaggle/input/notebooks/dorast/vae-b/runs/baseline_check/checkpoints/vae_domain_b_epoch0024.pt',
# ])

# result.check_returncode()


In [22]:
import subprocess
result = subprocess.run([
    'python', 'train_vae_domain_b.py',
    '--clean-dir', VOC_JPEG_DIR, '--val-clean-dir', VAL_SUBSET_DIR,
    '--masks-dir', *MASKS_DIRS, '--val-masks-dir', *MASKS_DIRS,
    '--epochs', '45', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
    '--n-downsample', '3', '--latent-channels', '128',
    '--kl-weight', '0.02', '--kl-anneal-epochs', '8',
    '--patience', '6', '--stop-after-epoch', '20',
    '--val-every', '1', '--log-every', '20', '--sample-every', '50', '--save-every', '4',
    '--amp', '--out-dir', './runs/vae_domain_b_kl002_anneal', '--device', 'cuda',
    '--resume', '/kaggle/input/notebooks/dorast/vae-b/runs/vae_domain_b_kl002_anneal/checkpoints/best.pt',
])
result.check_returncode()

[2026-09-13 15:36:22] Using device: cuda
[2026-09-13 15:36:22]   GPU: Tesla T4
[2026-09-13 15:36:22]   CPUs available: 4, --num-workers set to 2
[2026-09-13 15:36:22] Building dataset index...
[2026-09-13 15:36:23] Loaded 17125 clean images, 12000 masks (blend_mode=screen)
[2026-09-13 15:36:23] Fetching a fixed sample batch for visualization...
[2026-09-13 15:36:23] Dataset ready.
[2026-09-13 15:36:23] Building validation dataset from /kaggle/working/voc_val_subset/images / ['/kaggle/input/datasets/dorast/generated-masks/dirt', '/kaggle/input/datasets/dorast/generated-masks/scratches', '/kaggle/input/datasets/dorast/generated-masks/smut', '/kaggle/input/datasets/dorast/generated-masks/spots']...
[2026-09-13 15:36:23] Loaded 150 validation clean images
[2026-09-13 15:36:24] Resuming from /kaggle/input/notebooks/dorast/vae-b/runs/vae_domain_b_kl002_anneal/checkpoints/best.pt
[2026-09-13 15:36:27] Model has 42,294,531 parameters
[2026-09-13 15:36:27] Starting training: epochs 15-45


Epoch 15/45:   2%|▏         | 19/1070 [00:06<04:08,  4.23batch/s, loss=0.3219]

[2026-09-13 15:36:34]   step 15000: data_time=0.000s compute_time=0.235s
[2026-09-13 15:36:34]   Saved sample grid: ./runs/vae_domain_b_kl002_anneal/samples/step_0015000.png


Epoch 15/45:   4%|▎         | 40/1070 [00:11<04:04,  4.21batch/s, loss=0.3774]

[2026-09-13 15:36:39]   step 15020: data_time=0.000s compute_time=0.238s


...

Epoch 21/45:  98%|█████████▊| 1049/1070 [05:24<00:06,  3.30batch/s, loss=0.3664]

[2026-09-13 16:14:52]   Saved sample grid: ./runs/vae_domain_b_kl002_anneal/samples/step_0022450.png


Epoch 21/45:  99%|█████████▉| 1060/1070 [05:27<00:03,  3.27batch/s, loss=0.4298]

[2026-09-13 16:14:55]   step 22460: data_time=0.000s compute_time=0.303s


[2026-09-13 16:14:58] [Epoch 21/45] loss=0.3789 recon_clean=0.1148 recon_degraded=0.1152 consistency=0.0555 kl=2.3342 kl_weight=0.02000 lr=0.000200 epoch_time=5m 30s total_elapsed=38m 30s
[2026-09-13 16:15:01]   [Validation] epoch 21: loss=0.3979 recon_clean=0.1148 recon_degraded=0.1218 consistency=0.0609 kl=2.2333
[2026-09-13 16:15:01]   Validation loss hasn't improved on the best value (0.3443) for 6 consecutive checks (>= --patience 6) at epoch 21 -- stopping early. This catches divergence even when reconstruction quality (recon_clean/recon_degraded) looks fine, since it's driven by the combined validation loss, not just one component.
[2026-09-13 16:15:01]   Saved final checkpoint before stopping: ./runs/vae_domain_b_kl002_anneal/checkpoints/vae_domain_b_epoch0021.pt
[2026-09-13 16:15:01]   Note: best.pt (epoch with val_loss=0.3443) is likely more useful than this final checkpoint for downstream use, given training had stopped improving.
[2026-09-13 16:15:01] Training complete. Tot

## 10. Full training run

In [23]:
# Example full run:
# !python train_vae_domain_b.py \
#     --clean-dir "$VOC_JPEG_DIR" --val-clean-dir "$VAL_SUBSET_DIR" \
#     --masks-dir $MASKS_DIRS --val-masks-dir $MASKS_DIRS \
#     --epochs 50 --batch-size 16 --image-size 128 --num-workers 2 \
#     --n-downsample 3 --latent-channels 64 \
#     --val-every 1 --amp --out-dir ./runs/vae_domain_b --device cuda

# To resume from a previous session's checkpoint:
# !python train_vae_domain_b.py \
#     --clean-dir "$VOC_JPEG_DIR" --val-clean-dir "$VAL_SUBSET_DIR" \
#     --masks-dir $MASKS_DIRS --val-masks-dir $MASKS_DIRS \
#     --epochs 50 --batch-size 16 --image-size 128 \
#     --n-downsample 3 --latent-channels 64 --val-every 1 \
#     --amp --out-dir ./runs/vae_domain_b --device cuda \
#     --resume /kaggle/working/runs/vae_domain_b/checkpoints/<latest>.pt


In [24]:
import shutil

for folder in ['/kaggle/working/voc_data']:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
        print(f'Deleted: {folder}')
    else:
        print(f'Not found (already clean): {folder}')

Deleted: /kaggle/working/voc_data
